# 09 Publication-Quality Figures and Tables

Publication artifacts for the revised evaluation protocols.

- Broaching: fixed train/validation/test split, bootstrap CIs on the independent test set.
- Turning: run-wise grouped five-fold CV, reported as mean +/- SD across outer folds.
- Historical turning fixed-split and grayscale baseline artifacts are not used.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.metrics import precision_recall_curve, average_precision_score

REPO_ROOT = Path("..").resolve()
REPORT_DIR = REPO_ROOT / "reports"
PUB_DIR = REPORT_DIR / "publication"
FIG_DIR = PUB_DIR / "figures"
TAB_DIR = PUB_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# Broaching fixed-split artifacts
BROACH_AE_SCORES = REPORT_DIR / "scores" / "ae_scores_broach_dataset.csv"
BROACH_AE_CI = REPORT_DIR / "tables" / "metrics_broaching_ae_fixedsplit_with_ci.csv"
BROACH_BASE_SCORES = REPORT_DIR / "baselines" / "baseline_scores_broaching_fixedsplit_rgb.csv"
BROACH_BASE_CI = REPORT_DIR / "tables" / "metrics_broaching_baselines_fixedsplit_rgb_with_ci.csv"

# Turning grouped-CV artifacts
TURNING_ASSIGNMENTS = REPORT_DIR / "cv" / "turning_ae_grouped_cv_assignments.csv"
TURNING_AE_METRICS = REPORT_DIR / "tables" / "metrics_turning_ae_grouped_cv.csv"
TURNING_BASE_METRICS = REPORT_DIR / "tables" / "metrics_turning_grouped_cv_rgb.csv"
TURNING_AE_OOF = REPORT_DIR / "scores" / "ae_scores_turning_grouped_cv.csv"
TURNING_BASE_OOF = REPORT_DIR / "baselines" / "baseline_scores_turning_grouped_cv_rgb.csv"

BASE_LABELS = {
    "one_class_svm_rgb_features": "One-Class SVM",
    "isolation_forest_rgb_features": "Isolation Forest",
    "pca_rgb_reconstruction": "PCA Reconstruction",
}
TURN_AE_LABELS = {
    "global_mse": "CNN-AE Global MSE",
    "global_mae": "CNN-AE Global MAE",
    "sve_max": "CNN-AE SVE Max",
    "sve_topk": "CNN-AE SVE Top-k",
}
BROACH_AE_LABELS = {
    "global_mse": "Global MSE", "global_mae": "Global MAE",
    "sve_5": "SVE (5 Segments)", "sve_7": "SVE (7 Segments)",
}

def save_figure(fig, stem):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(FIG_DIR / f"{stem}.{ext}", bbox_inches="tight", dpi=600 if ext=="png" else None)
    plt.close(fig)

def mean_sd(df, group):
    return df.groupby(group).agg(
        pr_auc_mean=("pr_auc","mean"), pr_auc_std=("pr_auc","std"),
        f1_mean=("f1","mean"), f1_std=("f1","std"),
        precision_mean=("precision","mean"), precision_std=("precision","std"),
        recall_mean=("recall","mean"), recall_std=("recall","std"),
    ).reset_index()

def ipr(y, scores):
    p,r,_=precision_recall_curve(y,scores)
    order=np.argsort(r); r=r[order]; p=p[order]
    ru=np.unique(r); pu=np.array([p[r==v].max() for v in ru])
    return ru, np.maximum.accumulate(pu[::-1])[::-1]


## Turning grouped-CV protocol and fold counts

This replaces the historical 472/104/75 turning fixed-split presentation.


In [2]:
a=pd.read_csv(TURNING_ASSIGNMENTS)
assert a["source_run"].notna().all() and a["cv_fold"].between(1,5).all()
fold_counts=a.groupby(["cv_fold","label"]).size().unstack(fill_value=0)
fold_counts.to_csv(TAB_DIR / "turning_grouped_cv_fold_counts.csv")
print("samples:",len(a),"runs:",a["source_run"].nunique())
display(fold_counts)

fig,ax=plt.subplots(figsize=(7,2.5)); ax.set_axis_off(); ax.set_xlim(0,1); ax.set_ylim(0,1)
def box(x,title,sub,color):
    p=FancyBboxPatch((x,.36),.14,.30,boxstyle="round,pad=.02",facecolor=color,edgecolor=".3")
    ax.add_patch(p); ax.text(x+.07,.55,title,ha="center",weight="bold",fontsize=8); ax.text(x+.07,.43,sub,ha="center",fontsize=6.5)
xs=[.02,.19,.36,.53,.70,.86]
items=[("Time series","turning data","#E8EEF7"),("Segmentation","2.5-s windows","#E8EEF7"),("Proxy labels","chatter/nominal","#F6E2B8"),("RGB spectra","X/Y/Z","#D7E8D4"),("Run grouping","source_run","#E3D7F0"),("5-fold CV","run-wise","#F1C7C2")]
for x,(t,s,c) in zip(xs,items): box(x,t,s,c)
for x0,x1 in zip([.16,.33,.50,.67,.84],[.19,.36,.53,.70,.86]):
    ax.add_patch(FancyArrowPatch((x0,.51),(x1,.51),arrowstyle="-|>",mutation_scale=9,color=".3"))
ax.text(.5,.84,"Turning dataset: run-wise grouped evaluation protocol",ha="center",fontsize=9,weight="bold")
ax.text(.5,.15,"Held-out machining runs form each outer evaluation fold; development runs are split into one-class fitting and threshold calibration.",ha="center",fontsize=6.8)
save_figure(fig,"turning_grouped_cv_protocol")


samples: 651 runs: 63


label,chatter,no_chatter
cv_fold,,
1,15,121
2,11,124
3,10,118
4,11,109
5,14,118


## Turning performance: same grouped folds for CNN-AE and RGB baselines


In [3]:
if not TURNING_AE_METRICS.exists(): raise FileNotFoundError(f"Missing {TURNING_AE_METRICS}; export notebook 05 first.")
if not TURNING_BASE_METRICS.exists(): raise FileNotFoundError(TURNING_BASE_METRICS)
ae=pd.read_csv(TURNING_AE_METRICS); base=pd.read_csv(TURNING_BASE_METRICS)
assert ae["outer_fold"].nunique()==5 and base["outer_fold"].nunique()==5
aes=mean_sd(ae,"score"); aes["Method"]=aes["score"].map(TURN_AE_LABELS)
bs=mean_sd(base,"method"); bs["Method"]=bs["method"].map(BASE_LABELS)
cols=["Method","pr_auc_mean","pr_auc_std","f1_mean","f1_std","precision_mean","precision_std","recall_mean","recall_std"]
turning=pd.concat([aes[cols],bs[cols]],ignore_index=True)
turning.to_csv(TAB_DIR / "turning_grouped_cv_method_comparison.csv",index=False)
turning.to_latex(TAB_DIR / "turning_grouped_cv_method_comparison.tex",index=False,float_format="%.3f")
display(turning)

fig,ax=plt.subplots(figsize=(6.8,2.6)); x=np.arange(len(turning)); w=.34
ax.bar(x-w/2,turning.pr_auc_mean,w,yerr=turning.pr_auc_std,capsize=2,label="PR-AUC")
ax.bar(x+w/2,turning.f1_mean,w,yerr=turning.f1_std,capsize=2,label="F1-score")
ax.set_xticks(x); ax.set_xticklabels(turning.Method,rotation=25,ha="right"); ax.set_ylim(0,1.1); ax.set_ylabel("Score")
ax.set_title("Turning: performance across run-wise grouped outer folds"); ax.grid(axis="y",color=".9"); ax.legend(frameon=False)
save_figure(fig,"turning_grouped_cv_performance")

# Fold-wise threshold decisions summarized as OOF confusion totals
rows=[]
for score,label in TURN_AE_LABELS.items():
    g=ae[ae.score==score]; rows.append({"Method":label,"TN":g.tn.sum(),"FP":g.fp.sum(),"FN":g.fn.sum(),"TP":g.tp.sum()})
for m,label in BASE_LABELS.items():
    g=base[base.method==m]; rows.append({"Method":label,"TN":g.tn.sum(),"FP":g.fp.sum(),"FN":g.fn.sum(),"TP":g.tp.sum()})
pd.DataFrame(rows).to_csv(TAB_DIR / "turning_grouped_cv_confusion_totals.csv",index=False)


,Method,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,CNN-AE Global MAE,0.813061,0.202238,0.655596,0.210514,0.650504,0.188682,0.748831,0.319049
1,CNN-AE Global MSE,0.796922,0.210110,0.655029,0.201112,0.692079,0.180530,0.734545,0.330377
2,CNN-AE SVE Max,0.754337,0.206530,0.603756,0.183844,0.631852,0.193713,0.700260,0.334004
3,CNN-AE SVE Top-k,0.768354,0.227199,0.633068,0.199414,0.688971,0.208015,0.720260,0.344305
4,Isolation Forest,0.881970,0.137520,0.746813,0.192833,0.878914,0.132052,0.716450,0.279398
5,One-Class SVM,0.753695,0.172332,0.646937,0.169438,0.701966,0.230812,0.742165,0.315429
6,PCA Reconstruction,0.723586,0.192068,0.613999,0.165428,0.661354,0.233866,0.708831,0.307694


## Turning OOF precision-recall curves (optional)

Only generated when the new grouped-CV OOF score files exist. Historical fixed-split scores are never used as fallback.


In [4]:
if TURNING_AE_OOF.exists() and TURNING_BASE_OOF.exists():
    print("Grouped-CV OOF score files found. Add pooled OOF PR curves if desired.")
else:
    print("Grouped-CV OOF score files not available; skipping Turning OOF PR curves.")


Grouped-CV OOF score files found. Add pooled OOF PR curves if desired.


## Broaching fixed-split comparison


In [5]:
if not BROACH_AE_CI.exists(): raise FileNotFoundError(BROACH_AE_CI)
if not BROACH_BASE_CI.exists(): raise FileNotFoundError(BROACH_BASE_CI)
bae=pd.read_csv(BROACH_AE_CI); bb=pd.read_csv(BROACH_BASE_CI); rows=[]
for _,r in bb.iterrows(): rows.append({"Method":BASE_LABELS.get(r.method,r.method),"PR-AUC":r.pr_auc,"F1-score":r.f1,"Precision":r.precision,"Recall":r.recall})
for _,r in bae.iterrows():
    if r.score in BROACH_AE_LABELS: rows.append({"Method":BROACH_AE_LABELS[r.score],"PR-AUC":r.pr_auc,"F1-score":r.f1,"Precision":r.precision,"Recall":r.recall})
broach=pd.DataFrame(rows)
broach.to_csv(TAB_DIR / "broaching_fixedsplit_method_comparison.csv",index=False)
broach.to_latex(TAB_DIR / "broaching_fixedsplit_method_comparison.tex",index=False,float_format="%.3f")
display(broach)


,Method,PR-AUC,F1-score,Precision,Recall
0,Isolation Forest,0.640653,0.530612,0.481481,0.590909
1,One-Class SVM,0.838384,0.700000,0.777778,0.636364
2,PCA Reconstruction,0.982175,0.842105,1.000000,0.727273
3,Global MSE,0.592931,0.666667,1.000000,0.500000
4,Global MAE,0.465159,0.516129,0.888889,0.363636
5,SVE (5 Segments),0.945071,0.909091,0.909091,0.909091
6,SVE (7 Segments),0.979270,0.850000,0.944444,0.772727


## Broaching precision-recall curves

All methods use the same independent fixed test split.


In [10]:
bscore=pd.read_csv(BROACH_BASE_SCORES); ascore=pd.read_csv(BROACH_AE_SCORES); at=ascore[ascore["split"]=="test"].copy()
y=(at["target"] if "target" in at.columns else at["label"]).astype(int).to_numpy()
fig,ax=plt.subplots(figsize=(4.2,3.6))
for m,label in BASE_LABELS.items():
    g=bscore[bscore.method==m].copy()
    if "split" in g.columns: g=g[g.split=="test"]
    yy=g.target.astype(int).to_numpy(); ss=g.score_value.to_numpy(); r,p=ipr(yy,ss); ap=average_precision_score(yy,ss)
    ax.step(r,p,where="post",linewidth=1.6,label=f"{label} ({ap:.3f})")
for s in ["sve_5","sve_7"]:
    if s not in at.columns: continue
    ss=at[s].to_numpy(); r,p=ipr(y,ss); ap=average_precision_score(y,ss)
    ax.step(r,p,where="post",linewidth=1.6,label=f"{BROACH_AE_LABELS[s]} ({ap:.3f})")
ax.set(xlabel="Recall",ylabel="Precision",xlim=(0,1.02),ylim=(0,1.02))
ax.grid(
    color=".9"
)

ax.legend(
    frameon=False,
    fontsize=8,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
)

fig.subplots_adjust(
    top=0.72
)

save_figure(
    fig,
    "broaching_fixedsplit_pr_curves"
)


## CNN-AE architecture (PyTorch implementation)


In [7]:
layers=pd.DataFrame([
["Input","Input","100 x 150 x 3",0],["Encoder","Conv2d 3->4, 3x3 + ReLU","100 x 150 x 4",112],["Encoder","MaxPool2d 2x2","50 x 75 x 4",0],["Encoder","Conv2d 4->8, 3x3 + ReLU","50 x 75 x 8",296],["Encoder","MaxPool2d 2x3","25 x 25 x 8",0],["Encoder","Conv2d 8->12, 3x3 + ReLU","25 x 25 x 12",876],["Bottleneck","Flatten + Linear 7500->16 + ReLU","16",120016],["Decoder","Linear 16->7500 + ReLU","7500",127500],["Decoder","Conv/Upsample stack","100 x 150 x 4",2472],["Output","Conv2d 4->3, 3x3 + Sigmoid","100 x 150 x 3",111]],columns=["Stage","Layer","Output shape","Parameters"])
layers.to_csv(TAB_DIR / "cnn_ae_architecture_publication.csv",index=False); layers.to_latex(TAB_DIR / "cnn_ae_architecture_publication.tex",index=False)
print("Total parameters:",int(layers.Parameters.sum())); display(layers)


Total parameters: 251383


,Stage,Layer,Output shape,Parameters
0,Input,Input,100 x 150 x 3,0
1,Encoder,"Conv2d 3->4, 3x3 + ReLU",100 x 150 x 4,112
2,Encoder,MaxPool2d 2x2,50 x 75 x 4,0
3,Encoder,"Conv2d 4->8, 3x3 + ReLU",50 x 75 x 8,296
4,Encoder,MaxPool2d 2x3,25 x 25 x 8,0
5,Encoder,"Conv2d 8->12, 3x3 + ReLU",25 x 25 x 12,876
6,Bottleneck,Flatten + Linear 7500->16 + ReLU,16,120016
7,Decoder,Linear 16->7500 + ReLU,7500,127500
8,Decoder,Conv/Upsample stack,100 x 150 x 4,2472
9,Output,"Conv2d 4->3, 3x3 + Sigmoid",100 x 150 x 3,111


## Artifact index


In [8]:
arts=sorted([p.relative_to(REPO_ROOT).as_posix() for p in PUB_DIR.rglob("*") if p.is_file()])
idx=pd.DataFrame({"artifact":arts}); idx.to_csv(PUB_DIR / "artifact_index.csv",index=False); display(idx)


,artifact
0,reports/publication/artifact_index.csv
1,reports/publication/figures/baseline_confusion...
2,reports/publication/figures/baseline_confusion...
3,reports/publication/figures/baseline_confusion...
4,reports/publication/figures/baseline_metric_pa...
...,...
75,reports/publication/tables/turning_grouped_cv_...
76,reports/publication/tables/turning_method_comp...
77,reports/publication/tables/turning_method_comp...
78,reports/publication/tables/turning_split_count...
